# Flower Classification - Data Preprocessing
## Data Loading, Cleaning, and Preparation

This notebook handles data loading, cleaning, and preprocessing for flower classification.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.datasets import load_iris
import warnings
warnings.filterwarnings('ignore')
sns.set_style('whitegrid')

## 1. Data Loading

In [ ]:
# Load Iris dataset from CSV file
print('Loading Iris dataset from CSV...')
df = pd.read_csv('../data/raw/iris.csv')

# Rename columns to match sklearn format
df.columns = ['sepal_length', 'sepal_width', 'petal_length', 'petal_width', 'species']

# Encode species to numeric
species_map = {'setosa': 0, 'versicolor': 1, 'virginica': 2}
df['species_id'] = df['species'].map(species_map)

print(f'Data shape: {df.shape}')
print(f'Features: sepal_length, sepal_width, petal_length, petal_width')
print(f'Classes: {df["species"].unique()}')
print(f'Class distribution:\n{df["species"].value_counts()}')
df.head()

## 2. Data Cleaning

In [ ]:
# Check for missing values
print('Missing values per column:')
print(df.isnull().sum())

In [ ]:
# Check for duplicates
duplicates = df.duplicated().sum()
print(f'Number of duplicate rows: {duplicates}')

# Drop duplicates if any
if duplicates > 0:
    df = df.drop_duplicates()
    print(f'Dropped {duplicates} duplicates')

# Drop rows with missing values
df = df.dropna()
print('Dropped rows with missing values')

## 3. Data Exploration

In [ ]:
# Basic statistics
print('Dataset Statistics:')
df.describe().T

In [ ]:
# Class distribution
print('Class Distribution:')
print(df['species_name'].value_counts())

plt.figure(figsize=(8, 5))
sns.countplot(data=df, x='species_name', palette='Set2')
plt.title('Class Distribution')
plt.xlabel('Species')
plt.ylabel('Count')
plt.show()

## 4. Feature and Target Separation

In [ ]:
# Separate features and target
feature_columns = df.select_dtypes(include=[np.number]).columns.tolist()
if 'species' in feature_columns:
    feature_columns.remove('species')

X = df[feature_columns]
y = df['species']

print(f'Features: {feature_columns}')
print(f'X shape: {X.shape}')
print(f'y shape: {y.shape}')

## 5. Train-Test Split

In [ ]:
# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Train size: {len(X_train)}')
print(f'Test size: {len(X_test)}')
print(f'Train class distribution:\n{y_train.value_counts()}')
print(f'Test class distribution:\n{y_test.value_counts()}')

## 6. Feature Scaling

In [ ]:
# Fit scaler on training data and transform
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print('Fitted and transformed features')
print(f'Scaled X_train shape: {X_train_scaled.shape}')
print(f'Scaled X_test shape: {X_test_scaled.shape}')

In [ ]:
# Compare before and after scaling
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Before scaling
sns.boxplot(data=X_train, ax=axes[0])
axes[0].set_title('Before Scaling')
axes[0].set_xlabel('Features')
axes[0].tick_params(axis='x', rotation=45)

# After scaling
sns.boxplot(data=X_train_scaled, ax=axes[1])
axes[1].set_title('After Scaling')
axes[1].set_xlabel('Features')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

## 7. Target Encoding

In [ ]:
# Encode target labels
label_encoder = LabelEncoder()
y_train_encoded = label_encoder.fit_transform(y_train)
y_test_encoded = label_encoder.transform(y_test)

print(f'Encoded classes: {label_encoder.classes_}')
print(f'y_train_encoded shape: {y_train_encoded.shape}')
print(f'y_test_encoded shape: {y_test_encoded.shape}')

In [ ]:
# Verify decoding works
y_decoded = label_encoder.inverse_transform(y_train_encoded)
print(f'Original labels: {y_train.values[:10]}')
print(f'Decoded labels: {y_decoded[:10]}')

## 8. Save Preprocessed Data

In [ ]:
# Save preprocessed data for use in other notebooks
import joblib
from pathlib import Path

# Create directories if they don't exist
data_dir = Path('../data/processed')
data_dir.mkdir(parents=True, exist_ok=True)

# Save preprocessed data
np.save(data_dir / 'X_train.npy', X_train_scaled)
np.save(data_dir / 'X_test.npy', X_test_scaled)
np.save(data_dir / 'y_train.npy', y_train_encoded)
np.save(data_dir / 'y_test.npy', y_test_encoded)

# Save preprocessing objects
models_dir = Path('../models/artifacts')
models_dir.mkdir(parents=True, exist_ok=True)

joblib.dump(scaler, models_dir / 'scaler.joblib')
joblib.dump(label_encoder, models_dir / 'label_encoder.joblib')
joblib.dump(feature_columns, models_dir / 'feature_columns.joblib')

print('Saved preprocessed data and artifacts!')
print(f'  - X_train.npy, X_test.npy')
print(f'  - y_train.npy, y_test.npy')
print(f'  - scaler.joblib, label_encoder.joblib')
print(f'  - feature_columns.joblib')

## Summary

✅ Loaded Iris dataset (150 samples, 4 features, 3 classes)  
✅ Cleaned data (no missing values or duplicates found)  
✅ Split data: 80% train (120), 20% test (30) with stratification  
✅ Applied StandardScaler for feature normalization  
✅ Encoded target labels using LabelEncoder  
✅ Saved preprocessed data and artifacts for model training